# DATOS

“El dataset base proviene de Make Music Equal (licencia CC BY-SA 4.0).
Este fue enriquecido con datos obtenidos mediante la API de Chartmetric, los cuales poseen restricciones de uso.
Por este motivo, las variables derivadas de dicha fuente fueron transformadas y agregadas de forma tal que no permiten reconstruir los valores originales.”

PASO 1: Base de artistas para obtener IDs antes de conectar a la API
Fuente: Make Music Equal (CC BY-SA 4.0)
La base de datos de artistas se descargó del dataset de acceso público MakeMusicEqual versión 3/3/2026: 936420, 10.
https://chartmetric-public.s3.us-west-2.amazonaws.com/make-music-equal/mme_artist_info.csv 
Se eliminaron registros duplicados [nombre + url de artista]: 172
Resultado: 936248 filas/artistas. 
Legal: This work is licensed under a Creative Commons Attribution-ShareAlike 4.0 International License.

PASO 2: Construcción de las cohortes de datos a extraer
Base de datos MakeMusicEqual ordenada por Chartmetric_rank (rank 1 a 936248)
-cohorte 1: 1400 registros 
-cohorte 2: 600 registros 

Notas:
Hay discontinuidades en el atributo Chartmetric_rank (faltan números en la secuencia) porque la base makemusicequal está actualizada al 3/3/26 aunque fueron descargados el 23/3/2026. Misma fecha de acceso a la API. Como la variable chartmetrick_rank es dinámica, esa diferencia de 20 días hace que un artista pueda tener rank = 5 el 3/3/2026 y rank = 9 el 23/3/26. Sin embargo, eso no invalida la selección del conjunto de artistas sobre la cual se construye la extracción de datos porque el objetivo fue definir un conjunto de artistas ordenados por popularidad (medida por una fuenta de datos confiable). Además la variable rank no será utilizada en el análisis. Y por último, se comprueba que el rank puede variar sutilmente pero el conjunto de artistas definidos es predominantemente coincidente. 

PASO 3: API. Extracción de datos
Límite 14 días, 100 request x día.
Se apunta a dos endpoints: artist metadaga y artist live events
La extracción de datos se fragmenta por cohortes.


In [78]:
# !pip install pandas numpy matplotlib seaborn scikit-learn jupyter


# shows 2022 - 2023

In [ ]:
import os
import time
import json
import requests
import pandas as pd
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================

INPUT_CSV = r"C:\Users\Silvana\Downloads\taller tesis 1\make_music_equal\dataset_2903.csv"
API_KEY_FILE = r"C:\Users\Silvana\Downloads\taller tesis 1\make_music_equal\jambase_api_key.txt"
OUTPUT_DIR = r"C:\Users\Silvana\Downloads\taller tesis 1\make_music_equal\jambase_historico_validacion"

BASE_URL = "https://www.jambase.com/jb-api/v1/events"

SAMPLE_SIZE = 30
RANDOM_STATE = 42
PER_PAGE = 100
SLEEP_SECONDS = 1.1

YEAR_WINDOWS = {
    "2024": ("2024-01-01", "2024-12-31"),
    "2025": ("2025-01-01", "2025-12-31"),
}

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# ============================================================
# HELPERS
# ============================================================

def load_api_key(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read().strip()

def normalize_bool_series(s):
    return s.astype(str).str.strip().str.lower().map({
        "true": True,
        "false": False
    })

def get_nested(d, keys, default=None):
    current = d
    for k in keys:
        if not isinstance(current, dict):
            return default
        current = current.get(k)
        if current is None:
            return default
    return current

def extract_country_name(event):
    country = get_nested(event, ["location", "address", "addressCountry", "name"])
    if country is not None:
        return country
    country = get_nested(event, ["location", "address", "addressCountry"])
    if isinstance(country, str):
        return country
    return None

def extract_capacity(event):
    return get_nested(event, ["location", "maximumAttendeeCapacity"])

def extract_city_name(event):
    city = get_nested(event, ["location", "address", "addressLocality"])
    if isinstance(city, dict):
        return city.get("name")
    return city

def extract_venue_name(event):
    return get_nested(event, ["location", "name"])

def extract_offer_stats(event):
    offers = event.get("offers", [])
    if not isinstance(offers, list):
        return {
            "offer_min_price": None,
            "offer_max_price": None,
            "offer_currency": None,
            "n_offers": 0
        }

    min_prices = []
    max_prices = []
    currencies = []

    for offer in offers:
        if not isinstance(offer, dict):
            continue
        ps = offer.get("priceSpecification", {})
        if not isinstance(ps, dict):
            continue

        min_p = ps.get("minPrice")
        max_p = ps.get("maxPrice")
        cur = ps.get("priceCurrency")

        if min_p is not None:
            min_prices.append(min_p)
        if max_p is not None:
            max_prices.append(max_p)
        if cur is not None:
            currencies.append(cur)

    return {
        "offer_min_price": min(min_prices) if min_prices else None,
        "offer_max_price": max(max_prices) if max_prices else None,
        "offer_currency": currencies[0] if currencies else None,
        "n_offers": len(offers)
    }

def search_events_by_artist_name(api_key, artist_name, date_from, date_to):
    all_events = []
    raw_pages = []
    page = 1

    while True:
        params = {
            "apikey": api_key,
            "artistName": artist_name,
            "eventDateFrom": date_from,
            "eventDateTo": date_to,
            "expandPastEvents": "true",
            "perPage": PER_PAGE,
            "page": page
        }

        r = requests.get(BASE_URL, params=params, timeout=60)

        try:
            payload = r.json()
        except Exception:
            payload = {"raw_text": r.text}

        raw_pages.append({
            "page": page,
            "http_status": r.status_code,
            "payload": payload
        })

        if r.status_code != 200:
            return {
                "ok": False,
                "events": all_events,
                "raw_pages": raw_pages
            }

        success = payload.get("success", False)
        if not success:
            return {
                "ok": False,
                "events": all_events,
                "raw_pages": raw_pages
            }

        page_events = payload.get("events", [])
        if not isinstance(page_events, list):
            page_events = []

        all_events.extend(page_events)

        pagination = payload.get("pagination", {})
        total_pages = pagination.get("totalPages")

        if total_pages is None:
            if len(page_events) < PER_PAGE:
                break
        else:
            if page >= total_pages:
                break

        page += 1
        time.sleep(SLEEP_SECONDS)

    return {
        "ok": True,
        "events": all_events,
        "raw_pages": raw_pages
    }

def flatten_events(row, year_label, events):
    out = []

    for event in events:
        offer_stats = extract_offer_stats(event)

        out.append({
            "chartmetric_id": row.get("chartmetric_id"),
            "chartmetric_rank": row.get("chartmetric_rank"),
            "artist_name_input": row.get("artist_name"),
            "country_name_input": row.get("country_name"),
            "year": year_label,
            "event_id": event.get("identifier"),
            "event_name": event.get("name"),
            "start_date": event.get("startDate"),
            "end_date": event.get("endDate"),
            "event_status": event.get("eventStatus"),
            "venue_name": extract_venue_name(event),
            "city_name": extract_city_name(event),
            "country_name_event": extract_country_name(event),
            "venue_capacity": extract_capacity(event),
            "n_offers": offer_stats["n_offers"],
            "offer_min_price": offer_stats["offer_min_price"],
            "offer_max_price": offer_stats["offer_max_price"],
            "offer_currency": offer_stats["offer_currency"]
        })

    return out

def aggregate_events(df_events):
    if df_events.empty:
        return pd.DataFrame(columns=[
            "chartmetric_id", "artist_name_input", "year",
            "n_shows", "n_countries", "n_cities",
            "n_shows_with_capacity", "total_capacity", "avg_venue_capacity",
            "n_shows_with_price", "avg_low_price", "avg_high_price"
        ])

    df = df_events.copy()

    df["venue_capacity"] = pd.to_numeric(df["venue_capacity"], errors="coerce")
    df["offer_min_price"] = pd.to_numeric(df["offer_min_price"], errors="coerce")
    df["offer_max_price"] = pd.to_numeric(df["offer_max_price"], errors="coerce")

    agg = (
        df.groupby(["chartmetric_id", "artist_name_input", "year"], dropna=False)
          .agg(
              n_shows=("event_id", "nunique"),
              n_countries=("country_name_event", lambda s: s.dropna().nunique()),
              n_cities=("city_name", lambda s: s.dropna().nunique()),
              n_shows_with_capacity=("venue_capacity", lambda s: s.notna().sum()),
              total_capacity=("venue_capacity", "sum"),
              avg_venue_capacity=("venue_capacity", "mean"),
              n_shows_with_price=("offer_min_price", lambda s: s.notna().sum()),
              avg_low_price=("offer_min_price", "mean"),
              avg_high_price=("offer_max_price", "mean"),
          )
          .reset_index()
    )

    return agg

# ============================================================
# LOAD INPUT
# ============================================================

api_key = load_api_key(API_KEY_FILE)

df = pd.read_csv(INPUT_CSV)

required_cols = ["artist_name", "has_live_data"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Faltan columnas requeridas: {missing_cols}")

for col in ["chartmetric_id", "chartmetric_rank", "country_name"]:
    if col not in df.columns:
        df[col] = pd.NA

df["has_live_data_norm"] = normalize_bool_series(df["has_live_data"])

df_live = df[df["has_live_data_norm"] == True].copy()

if df_live.empty:
    raise ValueError("No hay filas con has_live_data == True.")

sample_n = min(SAMPLE_SIZE, len(df_live))
df_sample = df_live.sample(n=sample_n, random_state=RANDOM_STATE).copy()

# ============================================================
# RUN HISTORICAL VALIDATION EXTRACTION
# ============================================================

all_event_rows = []
summary_rows = []
raw_checks = []

for i, row in df_sample.reset_index(drop=True).iterrows():
    artist_name = row["artist_name"]
    print(f"[{i+1}/{len(df_sample)}] {artist_name}")

    for year_label, (date_from, date_to) in YEAR_WINDOWS.items():
        result = search_events_by_artist_name(
            api_key=api_key,
            artist_name=artist_name,
            date_from=date_from,
            date_to=date_to
        )

        n_events = len(result["events"])

        summary_rows.append({
            "chartmetric_id": row.get("chartmetric_id"),
            "chartmetric_rank": row.get("chartmetric_rank"),
            "artist_name_input": artist_name,
            "country_name_input": row.get("country_name"),
            "year": year_label,
            "ok": result["ok"],
            "n_events": n_events
        })

        raw_checks.append({
            "chartmetric_id": row.get("chartmetric_id"),
            "artist_name_input": artist_name,
            "year": year_label,
            "ok": result["ok"],
            "raw_pages": result["raw_pages"]
        })

        if result["ok"] and n_events > 0:
            flat_rows = flatten_events(row, year_label, result["events"])
            all_event_rows.extend(flat_rows)

        time.sleep(SLEEP_SECONDS)

df_summary = pd.DataFrame(summary_rows)
df_events = pd.DataFrame(all_event_rows)
df_agg = aggregate_events(df_events)

# ============================================================
# SAVE OUTPUTS
# ============================================================

sample_path = os.path.join(OUTPUT_DIR, "sample_artists_has_live_data_true.csv")
summary_path = os.path.join(OUTPUT_DIR, "jambase_historico_summary.csv")
events_path = os.path.join(OUTPUT_DIR, "jambase_historico_events_flat.csv")
agg_path = os.path.join(OUTPUT_DIR, "jambase_historico_agg_by_artist_year.csv")
raw_path = os.path.join(OUTPUT_DIR, "jambase_historico_raw_checks.json")

df_sample.to_csv(sample_path, index=False)
df_summary.to_csv(summary_path, index=False)
df_events.to_csv(events_path, index=False)
df_agg.to_csv(agg_path, index=False)

with open(raw_path, "w", encoding="utf-8") as f:
    json.dump(raw_checks, f, ensure_ascii=False, indent=2)

print("\nArchivos generados:")
print(sample_path)
print(summary_path)
print(events_path)
print(agg_path)
print(raw_path)

# shows 2026 para muerto_disuelto
sobre base ya chequeada de chartmetric. esto es un segundo paso

In [1]:
import os
import time
import json
import requests
import pandas as pd
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================

INPUT_CSV = r"C:\Users\Silvana\Documents\Maestria_Exactas\taller_tesis_1\music_industry_social_live_code\sin_shows_2026_checked_clean.csv"
OUTPUT_CSV = INPUT_CSV

API_KEY_FILE = r"C:\Users\Silvana\Documents\Maestria_Exactas\taller_tesis_1\music_industry_social_live_code\jambase_api_key.txt"
RAW_JSON = r"C:\Users\Silvana\Documents\Maestria_Exactas\taller_tesis_1\music_industry_social_live_code\jambase_2026_raw_checks.json"

BASE_URL = "https://www.jambase.com/jb-api/v1/events"

DATE_FROM = "2026-01-01"
DATE_TO = "2026-12-31"
PER_PAGE = 100
SLEEP_SECONDS = 1.1

# ============================================================
# HELPERS
# ============================================================

def load_api_key(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read().strip()

def normalize_bool_series(s):
    return s.astype(str).str.strip().str.lower().map({
        "true": True,
        "false": False
    })

def extract_event_list(payload):
    events = payload.get("events", [])
    if isinstance(events, list):
        return events
    return []

def get_event_key(event):
    event_id = event.get("identifier")
    if event_id is not None:
        return f"id:{event_id}"

    event_name = event.get("name", "")
    start_date = event.get("startDate", "")
    venue_name = ""

    location = event.get("location")
    if isinstance(location, dict):
        venue_name = location.get("name", "")

    return f"fallback:{event_name}|{start_date}|{venue_name}"

def search_events_by_artist_name(api_key, artist_name, date_from, date_to):
    all_events = []
    raw_pages = []
    page = 1

    while True:
        params = {
            "apikey": api_key,
            "artistName": artist_name,
            "eventDateFrom": date_from,
            "eventDateTo": date_to,
            "expandPastEvents": "true",
            "perPage": PER_PAGE,
            "page": page
        }

        r = requests.get(BASE_URL, params=params, timeout=60)

        try:
            payload = r.json()
        except Exception:
            payload = {"raw_text": r.text}

        raw_pages.append({
            "page": page,
            "http_status": r.status_code,
            "payload": payload
        })

        if r.status_code != 200:
            return {
                "ok": False,
                "events": all_events,
                "raw_pages": raw_pages,
                "error_detail": f"http_{r.status_code}"
            }

        success = payload.get("success", False)
        if not success:
            error_code = payload.get("code")
            error_message = payload.get("message")
            return {
                "ok": False,
                "events": all_events,
                "raw_pages": raw_pages,
                "error_detail": f"api_error: {error_code} | {error_message}"
            }

        page_events = extract_event_list(payload)
        all_events.extend(page_events)

        pagination = payload.get("pagination", {})
        total_pages = pagination.get("totalPages")

        if total_pages is None:
            if len(page_events) < PER_PAGE:
                break
        else:
            if page >= total_pages:
                break

        page += 1
        time.sleep(SLEEP_SECONDS)

    dedup = {}
    for event in all_events:
        dedup[get_event_key(event)] = event

    return {
        "ok": True,
        "events": list(dedup.values()),
        "raw_pages": raw_pages,
        "error_detail": None
    }

# ============================================================
# LOAD INPUT
# ============================================================

api_key = load_api_key(API_KEY_FILE)
df = pd.read_csv(INPUT_CSV)

required_cols = ["chartmetric_id", "artist_name", "shows_2026", "has_live_data", "api_status"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Faltan columnas requeridas: {missing_cols}")

for col in [
    "n_shows_2026_jambase",
    "shows_2026_jambase",
    "has_live_data_jambase",
    "api_status_jambase"
]:
    if col not in df.columns:
        df[col] = pd.NA

df["shows_2026_norm"] = normalize_bool_series(df["shows_2026"])
df["has_live_data_norm"] = normalize_bool_series(df["has_live_data"])

mask_chartmetric_sin_shows = (
    df["shows_2026_norm"].isna() &
    (df["has_live_data_norm"] == False)
)

mask_chartmetric_error = df["api_status"].astype(str).str.startswith("error", na=False)

mask_target = mask_chartmetric_sin_shows | mask_chartmetric_error
mask_pending_jambase = df["api_status_jambase"].isna()

df_narrow = df.loc[mask_target & mask_pending_jambase].copy()

print("Total filas archivo entrada:", len(df))
print("Casos target para JamBase:", int(mask_target.sum()))
print("Pendientes de consultar en JamBase:", len(df_narrow))

if df_narrow.empty:
    raise ValueError("No hay artistas pendientes para consultar en JamBase.")

# ============================================================
# RAW CHECKS PREVIOS
# ============================================================

if os.path.exists(RAW_JSON):
    with open(RAW_JSON, "r", encoding="utf-8") as f:
        try:
            raw_checks = json.load(f)
        except Exception:
            raw_checks = []
else:
    raw_checks = []

# ============================================================
# LOOP
# ============================================================

for i, row in enumerate(df_narrow.itertuples(index=False), start=1):
    artist_id = row.chartmetric_id
    artist_name = row.artist_name

    print(f"[{i}/{len(df_narrow)}] {artist_name} ({artist_id})")

    row_mask = (df["chartmetric_id"] == artist_id) & (df["artist_name"] == artist_name)

    try:
        result = search_events_by_artist_name(
            api_key=api_key,
            artist_name=artist_name,
            date_from=DATE_FROM,
            date_to=DATE_TO
        )

        raw_checks.append({
            "chartmetric_id": artist_id,
            "artist_name": artist_name,
            "raw_pages": result["raw_pages"]
        })

        if result["ok"]:
            n_events = len(result["events"])

            if n_events > 0:
                df.loc[row_mask, "n_shows_2026_jambase"] = n_events
                df.loc[row_mask, "shows_2026_jambase"] = True
                df.loc[row_mask, "has_live_data_jambase"] = True
                df.loc[row_mask, "api_status_jambase"] = "ok"
                print(f"  n_shows_2026_jambase={n_events}")
            else:
                df.loc[row_mask, "n_shows_2026_jambase"] = pd.NA
                df.loc[row_mask, "shows_2026_jambase"] = pd.NA
                df.loc[row_mask, "has_live_data_jambase"] = False
                df.loc[row_mask, "api_status_jambase"] = "ok"
                print("  sin shows en 2026 según JamBase")
        else:
            df.loc[row_mask, "n_shows_2026_jambase"] = pd.NA
            df.loc[row_mask, "shows_2026_jambase"] = pd.NA
            df.loc[row_mask, "has_live_data_jambase"] = pd.NA
            df.loc[row_mask, "api_status_jambase"] = result["error_detail"]
            print(f"  ERROR: {result['error_detail']}")

    except Exception as e:
        raw_checks.append({
            "chartmetric_id": artist_id,
            "artist_name": artist_name,
            "raw_pages": [],
            "exception": str(e)
        })

        df.loc[row_mask, "n_shows_2026_jambase"] = pd.NA
        df.loc[row_mask, "shows_2026_jambase"] = pd.NA
        df.loc[row_mask, "has_live_data_jambase"] = pd.NA
        df.loc[row_mask, "api_status_jambase"] = f"error: {str(e)}"

        print(f"  ERROR: {e}")

    time.sleep(SLEEP_SECONDS)

    if i % 25 == 0:
        df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

        with open(RAW_JSON, "w", encoding="utf-8") as f:
            json.dump(raw_checks, f, ensure_ascii=False, indent=2)

        print(f"  Guardado parcial en: {OUTPUT_CSV}")

# ============================================================
# SAVE FINAL
# ============================================================

df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

with open(RAW_JSON, "w", encoding="utf-8") as f:
    json.dump(raw_checks, f, ensure_ascii=False, indent=2)

print("\nProceso terminado.")
print("Archivo CSV:", OUTPUT_CSV)
print("Archivo raw JSON:", RAW_JSON)

print("\nResumen api_status_jambase:")
print(df["api_status_jambase"].value_counts(dropna=False))

print("\nArtistas con shows 2026 en JamBase:", int((df["shows_2026_jambase"] == True).sum()))
print("Artistas sin shows 2026 en JamBase:", int((df["has_live_data_jambase"] == False).sum()))
print("Artistas con error en JamBase:", int((df["api_status_jambase"].notna() & (df["api_status_jambase"] != "ok")).sum()))

Total filas archivo entrada: 2310
Casos target para JamBase: 1806
Pendientes de consultar en JamBase: 1806
[1/1806] Mike Mascott (7669361)
  ERROR: http_400
[2/1806] PRISET (4243258)
  ERROR: http_400
[3/1806] Sunstroke Project (273176)
  ERROR: http_400
[4/1806] alex g online (10283819)
  ERROR: http_400
[5/1806] Thome (3632898)
  ERROR: http_400
[6/1806] Ka-Flame (3626647)
  ERROR: http_400


KeyboardInterrupt: 

# levanto dataset

In [89]:
df = pd.read_csv("mme_artist_info_DATASET.csv")
# df = pd.read_csv("mme_artist_info.csv", low_memory=False)

C:\Users\Silvana\AppData\Local\Temp\ipykernel_19452\2302022066.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("mme_artist_info_DATASET.csv")


In [82]:
df.columns

Index(['chartmetric_rank', 'chartmetric_id', 'artist_name', 'chartmetric_url',
       'country_name', 'composition', 'career_stage', 'pronouns', 'genre',
       'data_date'],
      dtype='object')

In [83]:
df.shape

(936420, 10)

In [84]:
df.head()

,chartmetric_rank,chartmetric_id,artist_name,chartmetric_url,country_name,composition,career_stage,pronouns,genre,data_date
0,1,214945,Bad Bunny,https://app.chartmetric.com/artist/214945,Puerto Rico,solo,superstar,he/him,reggaeton,2026-03-03
1,2,2762,Taylor Swift,https://app.chartmetric.com/artist/2762,United States,solo,superstar,she/her,pop,2026-03-03
2,3,3501,Bruno Mars,https://app.chartmetric.com/artist/3501,United States,solo,superstar,he/him,pop,2026-03-03
3,4,3479,Justin Bieber,https://app.chartmetric.com/artist/3479,Canada,solo,superstar,he/him,pop,2026-03-03
4,5,3852,The Weeknd,https://app.chartmetric.com/artist/3852,Canada,solo,superstar,he/him,r&b/soul,2026-03-03


In [85]:
df["composition"].unique()


array(['solo', 'group', 'unknown'], dtype=object)

In [86]:


df["career_stage"].value_counts()


career_stage
undiscovered    766574
developing      136101
mid-level        18664
mainstream       12828
superstar         1880
legendary          369
Name: count, dtype: int64

# busco un rank 186 modificado

In [87]:
import requests

REFRESH_TOKEN = "112E1dROGObsUx9hWKnrOxLQWrjDZ9YXwoByI9ynpN7QjtFY2Lq1E3uW7zvuIrpY"

# 1) token
r = requests.post(
    "https://api.chartmetric.com/api/token",
    json={"refreshtoken": REFRESH_TOKEN}
)
r.raise_for_status()
token = r.json()["token"]

headers = {"Authorization": f"Bearer {token}"}

# 2) pedir rank 186
url = "https://api.chartmetric.com/api/artist/cm_artist_rank/list"
params = {
    "min": 186,
    "max": 186,
    "limit": 10,
    "offset": 0
}

r = requests.get(url, headers=headers, params=params)

print("STATUS:", r.status_code)
print("TEXT:", r.text[:500])

r.raise_for_status()

data = r.json()["obj"]["data"]

if not data:
    print("No existe rank 186 en la API.")
else:
    artist = data[0]
    print("Rank:", artist.get("cpp_rank"))
    print("Chartmetric ID:", artist.get("chartmetric_artist_id"))
    print("Nombre:", artist.get("name"))

STATUS: 429
TEXT: Rate limit exceeded, retry in 2 hours


HTTPError: 429 Client Error: Too Many Requests for url: https://api.chartmetric.com/api/artist/cm_artist_rank/list?min=186&max=186&limit=10&offset=0

In [ ]:
import pandas as pd

CSV_PATH = r"C:\Users\Silvana\Downloads\taller tesis 1\make_music_equal\mme_artist_info_DATASET.csv"
CM_ID = 166   # reemplazá por el chartmetric_id que te devolvió la API

df = pd.read_csv(CSV_PATH)
df["chartmetric_id"] = pd.to_numeric(df["chartmetric_id"], errors="coerce")

match = df[df["chartmetric_id"] == CM_ID]

if match.empty:
    print("Ese chartmetric_id no aparece en tu base original.")
else:
    print(match[["chartmetric_rank", "chartmetric_id", "artist_name"]].to_string(index=False))

C:\Users\Silvana\AppData\Local\Temp\ipykernel_19452\1365454167.py:6: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_PATH)


chartmetric_rank  chartmetric_id        artist_name
             203             166 Christina Aguilera


# duplicados

In [90]:
import pandas as pd

# --- 1. CONFIGURACIÓN Y KEY ÚNICA SEGURA ---
# Usamos el ID porque el ID NUNCA es NaN y es la verdad absoluta de Chartmetric
df['unique_entity_key'] = df['chartmetric_id'].astype(str)

# --- 2. ANÁLISIS DE COLISIONES (Nombres iguales, diferentes URL, no son dupli) ---
# Agrupamos por nombre y contamos cuántas URLs distintas existen para ese nombre
analisis_urls = df.groupby('artist_name')['chartmetric_url'].nunique()
colisiones_identidad = analisis_urls[analisis_urls > 1]

# --- 3. IDENTIFICACIÓN DE DUPLICADOS TÉCNICOS ---
# Casos donde la combinación Nombre + URL se repite (deberían ser el mismo registro)
mask_duplicados_tecnicos = df.duplicated(subset=['unique_entity_key'], keep=False)
df_duplicados = df[mask_duplicados_tecnicos].copy()

# Separamos dentro de estos duplicados técnicos:
# A. Filas Espejo: Son copias exactas en todas las columnas
filas_espejo = df_duplicados[df_duplicados.duplicated(keep='first')]

# B. Filas Discrepantes: Mismo artista/URL pero algo cambia en las otras columnas
# Usamos keep=False para marcar todas las filas que tienen conflicto
discrepantes = df_duplicados[~df_duplicados.duplicated(subset=df.columns.difference(['unique_entity_key']), keep=False)]

# --- 4. REPORTE FINAL ---
print("=== INFORME DE CALIDAD DE DATOS ===")
print(f"1. Colisiones de nombre (mismo nombre diferente url): {len(colisiones_identidad)}")
print(f"2. Duplicados Técnicos (Mismo artista + URL): {df.duplicated('unique_entity_key').sum()}")
print(f"   -> Copias exactas (Espejos): {len(filas_espejo)}")
print(f"   -> Registros con datos contradictorios: {len(discrepantes)}")
print("-" * 35)

# Visualización de discrepancias si existen
if not discrepantes.empty:
    print("Muestra de registros con datos discrepantes (mismo artista, distintos valores):")
    print(discrepantes.sort_values(by='unique_entity_key').head(10))

=== INFORME DE CALIDAD DE DATOS ===
1. Colisiones de nombre (mismo nombre diferente url): 31943
2. Duplicados Técnicos (Mismo artista + URL): 172
   -> Copias exactas (Espejos): 172
   -> Registros con datos contradictorios: 0
-----------------------------------


In [91]:
# 1. Calculamos cuántos se van a ir
total_antes = len(df)
df_sindupli = df.drop_duplicates(keep='first').copy()
eliminados = total_antes - len(df_sindupli)

# 3. Verificación final mejorada
print(f"=== RESULTADO DE LA LIMPIEZA ===")
print(f"- Registros originales: {total_antes}")
print(f"- Espejos exactos eliminados: {eliminados}")
print(f"- Artistas con nombre NaN conservados: {df_sindupli['artist_name'].isna().sum()}")
print(f"- Tamaño final del dataset: {len(df_sindupli)} filas.")

=== RESULTADO DE LA LIMPIEZA ===
- Registros originales: 936420
- Espejos exactos eliminados: 172
- Artistas con nombre NaN conservados: 8
- Tamaño final del dataset: 936248 filas.


# emprolijo rank

hay baches. 

In [92]:
# Convertimos la columna a numérica, lo que no sea número se vuelve NaN
df_sindupli['chartmetric_rank'] = pd.to_numeric(df_sindupli['chartmetric_rank'], errors='coerce')

In [93]:
# Ordenamos de menor a mayor rank (el rank 1 arriba)
df_sindupli = df_sindupli.sort_values(by='chartmetric_rank', ascending=True).reset_index(drop=True)

In [94]:
# Buscamos qué valores NO se pudieron convertir a número
valores_no_numericos = df_sindupli[pd.to_numeric(df_sindupli['chartmetric_rank'], errors='coerce').isna()]['chartmetric_rank'].unique()

print("--- VALORES SOSPECHOSOS ENCONTRADOS ---")
if len(valores_no_numericos) > 0:
    print(f"Estos valores causaban el error: {valores_no_numericos}")
else:
    print("No se encontraron strings, el problema podría haber sido solo de tipos (int vs float).")

--- VALORES SOSPECHOSOS ENCONTRADOS ---
Estos valores causaban el error: [nan]


In [96]:
# Contamos cuántos NaN hay después de la conversión
nulos_count = df_sindupli['chartmetric_rank'].isna().sum()
total_filas = len(df_sindupli)

print(f"Total de registros: {total_filas}")
print(f"Registros sin Rank (NaN): {nulos_count}")
print(f"Porcentaje de nulos: {(nulos_count / total_filas) * 100:.2f}%")

Total de registros: 936248
Registros sin Rank (NaN): 4
Porcentaje de nulos: 0.00%


In [97]:
# Filtramos el DataFrame para mostrar solo las filas donde el rank es NaN
df_nulos = df_sindupli[df_sindupli['chartmetric_rank'].isna()]

print(f"--- DETALLE DE LOS {len(df_nulos)} REGISTROS CON VALOR NaN ---")
# Mostramos las columnas principales para que identifiques a los artistas
print(df_nulos[['artist_name', 'chartmetric_rank', 'chartmetric_id', 'chartmetric_url']])

--- DETALLE DE LOS 4 REGISTROS CON VALOR NaN ---
         artist_name  chartmetric_rank  chartmetric_id  \
936244  Jabbawockeez               NaN          573449   
936245  Jill & Julia               NaN          342683   
936246     Ammi Boyz               NaN          887992   
936247  Steeven Savy               NaN         3939422   

                                   chartmetric_url  
936244   https://app.chartmetric.com/artist/573449  
936245   https://app.chartmetric.com/artist/342683  
936246   https://app.chartmetric.com/artist/887992  
936247  https://app.chartmetric.com/artist/3939422  


## elimino 4 nan en rank

In [98]:
# 1. Contamos antes de borrar para la documentación
total_antes = len(df_sindupli)
nulos_a_borrar = df_sindupli['chartmetric_rank'].isna().sum()

# 2. Eliminamos las filas donde chartmetric_rank es NaN
df_sindupli = df_sindupli.dropna(subset=['chartmetric_rank']).copy()

# 3. Ordenamos por rank y reseteamos el índice
# Esto asegura que el registro físico 0 sea el Rank 1, el 1000 sea el 1001, etc.
df_sindupli = df_sindupli.sort_values(by='chartmetric_rank', ascending=True).reset_index(drop=True)

# 4. Verificación final
total_despues = len(df_sindupli)
print(f"Registros iniciales: {total_antes}")
print(f"Registros eliminados (NaN): {nulos_a_borrar}")
print(f"Registros finales: {total_despues}")
print(f"--- Dataset ordenado y listo para segmentación ---")

Registros iniciales: 936248
Registros eliminados (NaN): 4
Registros finales: 936244
--- Dataset ordenado y listo para segmentación ---


# consolido MMequal

In [108]:
#RESERVO MMequal sin dupli

df_mmequal_936244 = df_sindupli
print("df_mmequal_936244")
df_mmequal_936244.shape

df_mmequal_936244


(936244, 11)

# df_1400

In [109]:
import pandas as pd

# 1. Recortamos los primeros 1400 registros (Slicing de Python)
# Como df_sindupli ya está ordenado por rank y con index reseteado,
# esto nos garantiza los 1400 mejores artistas de tu lista.
df_1400 = df_sindupli.iloc[:1400].copy()

# 2. Definimos las columnas de interés para la auditoría de estratos
estratos = ['country_name', 'composition', 'career_stage', 'genre', 'pronouns']

print("--- AUDITORÍA DE VALORES ÚNICOS (BLOQUE 1-1400) ---")

for col in estratos:
    # Obtenemos los valores únicos
    uniques = df_1400[col].unique()
    # Contamos cuántos hay
    count = len(uniques)
    
    print(f"\nColumna: {col} ({count} categorías)")
    print(uniques)
    print("-" * 40)

# 3. Verificamos cuántas combinaciones únicas de estratos hay en este recorte
combinaciones = df_1400[estratos].drop_duplicates().shape[0]
print(f"\nTotal de 'perfiles' únicos (combinaciones de estratos): {combinaciones}")

# --- EXPORTACIÓN PARA LA API ---

# 4. Seleccionamos solo las columnas necesarias para tu proceso de descarga
columnas_interes = ['chartmetric_rank', 'chartmetric_id', 'artist_name', 'chartmetric_url']
df_1400_api = df_1400[columnas_interes]

# 5. Exportamos a un archivo .txt
# Usamos sep='\t' (tabulador) para evitar conflictos con comas en nombres de artistas
nombre_archivo = 'df_1400_api.txt'
df_1400_api.to_csv(nombre_archivo, sep='\t', index=False, encoding='utf-8')

print(f"\nArchivo '{nombre_archivo}' creado con éxito.")
print(f"Rango de Rank exportado: {df_1400_api['chartmetric_rank'].min()} a {df_1400_api['chartmetric_rank'].max()}")

--- AUDITORÍA DE VALORES ÚNICOS (BLOQUE 1-1400) ---

Columna: country_name (58 categorías)
['Puerto Rico' 'United States' 'Canada' 'Barbados' 'United Kingdom'
 'Colombia' 'France' 'India' 'Sweden' 'South Korea' 'Australia' 'Jamaica'
 'Mexico' 'South Africa' 'Senegal' 'Spain' 'Netherlands' 'Argentina'
 'Nigeria' 'Norway' 'Brazil' 'Ireland' nan 'Belgium' 'Thailand' 'Iceland'
 'Chile' 'Germany' 'Pakistan' 'Dominican Republic' 'New Zealand' 'Uruguay'
 'Venezuela' 'Italy' 'Panama' 'Denmark' 'Türkiye' 'Austria' 'Japan'
 'Indonesia' 'Guatemala' 'Democratic Republic of Congo' 'Kyrgyzstan'
 'Egypt' 'Poland' 'Cuba' 'Romania' 'Russian Federation'
 'United States Minor Outlying Islands' 'Vietnam' 'Ghana' 'Lebanon'
 'Guyana' 'Zimbabwe' 'Tanzania' 'Taiwan' 'Kazakhstan' 'Hungary']
----------------------------------------

Columna: composition (3 categorías)
['solo' 'group' 'unknown']
----------------------------------------

Columna: career_stage (5 categorías)
['superstar' 'legendary' 'mainstream' '

# df_1401_7000

In [114]:
import pandas as pd

# 1. Recortamos del registro 1400 al 7000 (Slicing de Python)
# Esto toma las filas desde el índice 1400 hasta el 5999.
df_1401_7000 = df_sindupli.iloc[1400:7000].copy()

# 2. Definimos las columnas de interés para la auditoría de estratos
estratos = ['country_name', 'composition', 'career_stage', 'genre', 'pronouns']

print("--- AUDITORÍA DE VALORES ÚNICOS (BLOQUE 1401-7000) ---")

for col in estratos:
    # Obtenemos los valores únicos
    uniques = df_1401_7000[col].unique()
    # Contamos cuántos hay
    count = len(uniques)
    
    print(f"\nColumna: {col} ({count} categorías)")
    print(uniques)
    print("-" * 40)

# 3. Verificamos cuántas combinaciones únicas de estratos hay en este recorte
combinaciones = df_1401_7000[estratos].drop_duplicates().shape[0]
print(f"\nTotal de 'perfiles' únicos (combinaciones de estratos) en este bloque: {combinaciones}")

# --- EXPORTACIÓN PARA LA API ---

# 4. Seleccionamos solo las columnas necesarias para tu proceso de descarga
columnas_interes = ['chartmetric_rank', 'chartmetric_id', 'artist_name', 'chartmetric_url']
df_1401_7000_api = df_1401_7000[columnas_interes]

# 5. Exportamos a un archivo .txt
nombre_archivo = 'df_1401_7000_api.txt'
df_1401_7000_api.to_csv(nombre_archivo, sep='\t', index=False, encoding='utf-8')

print(f"\nArchivo '{nombre_archivo}' creado con éxito.")
print(f"Rango de Rank exportado: {df_1401_7000_api['chartmetric_rank'].min()} a {df_1401_7000_api['chartmetric_rank'].max()}")

--- AUDITORÍA DE VALORES ÚNICOS (BLOQUE 1401-7000) ---

Columna: country_name (106 categorías)
['Brazil' 'United States' 'United Kingdom' 'Mexico' 'Argentina'
 'South Korea' 'Jamaica' 'Australia' 'South Africa' 'Puerto Rico'
 'Finland' 'Germany' 'Italy' 'Belgium' 'Colombia' 'Sweden' 'Spain'
 'Pakistan' 'Virgin Islands' 'Canada' 'France' nan 'India' 'Indonesia'
 'Venezuela' 'Japan' 'Netherlands' 'Nigeria' 'Monaco' 'Philippines'
 'Russian Federation' 'Cuba' 'Ghana' 'Türkiye' 'Panama' 'Ireland'
 'Denmark' 'Malaysia' 'Qatar' 'Israel' 'Romania' 'Saudi Arabia'
 'Hong Kong' 'Poland' 'Czech Republic' 'New Zealand' 'Norway' 'Tanzania'
 'Dominican Republic' 'Iraq' 'Chile' 'Egypt' 'Algeria' 'Morocco'
 'Palestine' 'Ukraine' 'Armenia' 'Peru' 'Ecuador' 'Kenya' 'Lithuania'
 'Singapore' 'Nicaragua' 'Austria' 'China' 'Guatemala' 'Thailand'
 'Cape Verde' 'Mali' 'Uruguay' 'Greece' 'Côte d\x92Ivoire' 'Switzerland'
 'Lebanon' 'Antigua and Barbuda' 'Iceland' 'Taiwan' 'Sint Maarten'
 'Vietnam' 'Albania' 'Cro

In [116]:
print(f"Último ID del bloque anterior: {df_1400['chartmetric_rank'].iloc[-1]}")
print(f"Primer ID del bloque nuevo:    {df_1401_7000['chartmetric_rank'].iloc[0]}")

Último ID del bloque anterior: 1704.0
Primer ID del bloque nuevo:    1705.0


In [117]:
print("--- VERIFICACIÓN DE TRANSICIÓN ---")
print(f"BLOQUE 1400 (Última fila): ID {df_1400['chartmetric_id'].iloc[-1]} | Rank: {df_1400['chartmetric_rank'].iloc[-1]}")
print(f"BLOQUE 1401_7000 (Primera fila): ID {df_1401_7000['chartmetric_id'].iloc[0]} | Rank: {df_1401_7000['chartmetric_rank'].iloc[0]}")

# Verificamos si hay un salto grande en el ranking
salto = df_1401_7000['chartmetric_rank'].iloc[0] - df_1400['chartmetric_rank'].iloc[-1]
print(f"Distancia de ranking entre bloques: {salto}")

--- VERIFICACIÓN DE TRANSICIÓN ---
BLOQUE 1400 (Última fila): ID 3917158 | Rank: 1704.0
BLOQUE 1401_7000 (Primera fila): ID 1103173 | Rank: 1705.0
Distancia de ranking entre bloques: 1.0
